# MMDet3D Evaluation Debug (Waymo-KITTI)

목적: mmdet3d `test.py`가 어떤 포맷을 기대하는지, 현재 Waymo-KITTI info에서 무엇이 부족한지, 평가 지표가 안 나오는 이유를 단계별로 확인합니다. 모든 셀은 빠르게 검증할 수 있도록 작은 출력만 합니다.

In [1]:
!python -V

Python 3.10.19


In [3]:
from pathlib import Path
import os

WAYMO_ROOT = Path('/home/018219422/OccFormerWithWaymoData/data/waymo_v1-3-1/kitti_format')
CFG = Path('/home/018219422/customRCNN/3d_detection/checkpoints/pointpillars_hv_secfpn_8xb6-160e_kitti-3d-car.py')
CKPT = Path('/home/018219422/customRCNN/3d_detection/checkpoints/hv_pointpillars_secfpn_6x8_160e_kitti-3d-car_20220331_134606-d42d15ed.pth')
VAL_INFO_RAW = Path('/home/018219422/customRCNN/3d_detection/data_waymo_kitti/kitti_format/waymo_infos_val.pkl')
VAL_INFO_MIN = Path('/home/018219422/customRCNN/3d_detection/data_waymo_kitti/kitti_format/waymo_infos_val.pkl')

In [7]:
infos.keys()

dict_keys(['data_list', 'metainfo'])

In [4]:
import pickle
import numpy as np

with open(VAL_INFO_RAW, 'rb') as f:
    infos = pickle.load(f)
print('\nraw type:', type(infos), 'len:', len(infos))
first = infos[0]
print('\nkeys:', first.keys())
print('\npoint_cloud:', first.get('point_cloud'))
print('\nannos keys:', first.get('annos', {}).keys())
print('\nannos name len:', len(first.get('annos', {}).get('name', [])))


raw type: <class 'dict'> len: 2


KeyError: 0

In [5]:
import pickle
import numpy as np

with open(VAL_INFO_MIN, 'rb') as f:
    infos = pickle.load(f)
print('\nraw type:', type(infos), 'len:', len(infos))
first = infos[0]
print('\nkeys:', first.keys())
print('\npoint_cloud:', first.get('point_cloud'))
print('\nannos keys:', first.get('annos', {}).keys())
print('\nannos name len:', len(first.get('annos', {}).get('name', [])))


raw type: <class 'dict'> len: 2


KeyError: 0

In [4]:
import inspect
import mmdet3d.datasets.det3d_dataset as dd

print('parse_data_info expects lidar_points/lidar_path and num_pts_feats:')
print('\n'.join(inspect.getsource(dd.Det3DDataset.parse_data_info).split('\n')[:40]))

parse_data_info expects lidar_points/lidar_path and num_pts_feats:
    def parse_data_info(self, info: dict) -> dict:
        """Process the raw data info.

        Convert all relative path of needed modality data file to
        the absolute path. And process the `instances` field to
        `ann_info` in training stage.

        Args:
            info (dict): Raw info dict.

        Returns:
            dict: Has `ann_info` in training stage. And
            all path has been converted to absolute path.
        """

        if self.modality['use_lidar']:
            info['lidar_points']['lidar_path'] = \
                osp.join(
                    self.data_prefix.get('pts', ''),
                    info['lidar_points']['lidar_path'])

            info['num_pts_feats'] = info['lidar_points']['num_pts_feats']
            info['lidar_path'] = info['lidar_points']['lidar_path']
            if 'lidar_sweeps' in info:
                for sweep in info['lidar_sweeps']:
                 

In [ ]:
# 최소 필드(lidar_points, 빈 instances)만 가진 ann_file 생성
import pickle, os

with open(VAL_INFO_RAW, 'rb') as f:
    infos = pickle.load(f)
converted = []
for info in infos:
    pc = info.get('point_cloud', {})
    converted.append({
        'lidar_points': {
            'lidar_path': pc.get('velodyne_path', ''),
            'num_pts_feats': pc.get('num_features', 4),
        },
        'instances': [],  # GT 비어있어 mAP는 계산 안 됨
        'timestamp': info.get('timestamp', None),
    })

wrapped = {'metainfo': {'classes': ['Car']}, 'data_list': converted}
VAL_INFO_MIN.parent.mkdir(parents=True, exist_ok=True)
with open(VAL_INFO_MIN, 'wb') as f:
    pickle.dump(wrapped, f)
print('saved', VAL_INFO_MIN, 'len', len(converted))

## Run command (mAP 안 나오는 이유)
- `instances`가 비어 있어 GT가 없으므로 mAP 등 정량 지표는 계산되지 않음. 추론 결과만 저장 가능.
- mmdet3d test.py는 `--eval` 옵션을 받지 않는 버전이므로 빼야 함.
- 아래 명령을 셸에서 실행:
```bash
mim test mmdet3d \
  --config $CFG \
  --checkpoint $CKPT \
  --work-dir /home/018219422/customRCNN/3d_detection/results/waymo_pointpillars_val \
  --cfg-options \
    test_dataloader.dataset.data_root=$WAYMO_ROOT \
    test_dataloader.dataset.ann_file=/home/018219422/customRCNN/3d_detection/cache/kitti_infos_val_mmdet3d_v2.pkl \
    test_dataloader.dataset.data_prefix.pts=training/velodyne \
    test_evaluator.ann_file=/home/018219422/customRCNN/3d_detection/cache/kitti_infos_val_mmdet3d_v2.pkl \
    val_dataloader.dataset.data_root=$WAYMO_ROOT \
    val_dataloader.dataset.ann_file=/home/018219422/customRCNN/3d_detection/cache/kitti_infos_val_mmdet3d_v2.pkl \
    val_dataloader.dataset.data_prefix.pts=training/velodyne \
    train_dataloader.dataset.dataset.data_root=$WAYMO_ROOT \
    train_dataloader.dataset.dataset.data_prefix.pts=training/velodyne \
    test_dataloader.batch_size=1 \
    test_dataloader.num_workers=2
```
### 정량 평가를 원하면
- `annos`를 mmengine의 `instances` 포맷으로 매핑해 `gt_bboxes_3d`, `gt_labels_3d`를 생성해야 함.
- 또는 최신 `create_data.py waymo`로 다시 변환해 `lidar_points`/`instances`가 채워진 pkl을 생성.

In [2]:
from waymo_open_dataset import dataset_pb2 as open_dataset
import tensorflow as tf
tfrec = "/home/018219422/OccFormerWithWaymoData/data/waymo_v1-3-1/waymo_format/training/segment-10017090168044687777_6380_000_6400_000_with_camera_labels.tfrecord"
ds = tf.data.TFRecordDataset(tfrec, compression_type='')
cnt = 0
for raw in ds.take(3):
    frame = open_dataset.Frame()
    frame.ParseFromString(raw.numpy())
    cnt += 1
print("frames read:", cnt)

2025-12-03 13:48:58.210028: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-03 13:48:58.230222: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-03 13:48:58.372189: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-03 13:48:58.373418: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-03 13:48:59.439148: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT

frames read: 3
